# Component 3 - Phase 6: ROI selection (Sub-component 3.1)

Tile labels from rounds 1 + 2 (40 train tissues, 1,920 tiles): **analysable (good / usable) vs poor**.
Three methods on the same folds (lit-area rule, quality features + logistic regression, EfficientNet-B0);
pre-registered decision; ROI IoU on the 10 validation reference maps (target 0.75); then Phases 3-5
re-run **inside the ROI** (no new segmentation) and compared with the Phase 5 baseline.
Rules fixed in `config.yaml` (`roi_model:`).

**Runtime > Change runtime type > T4 or L4 GPU** (cells 3-4); the rest also runs on CPU.

In [ ]:
# 1. Setup (mount Drive, pull latest code, copy images, set paths)
from google.colab import drive; drive.mount("/content/drive")
REPO_DIR = "/content/drive/MyDrive/Colab Notebooks/Research_Project/xai-cornea-grading"
!git -C "{REPO_DIR}" pull --ff-only
%run "{REPO_DIR}/ml/cellular/colab_setup.py"

In [ ]:
# 2. Unit tests
!python -m pytest tests/test_roi.py tests/test_extraction.py tests/test_calibration.py tests/test_biomarkers.py -q

In [ ]:
# 3. Steps 6.1-6.3 - cross-validation of the three methods + pre-registered decision (~10-15 min on GPU)
!python -m roi.run --cv 2>&1 | grep -v -E "Warning|warn"

In [ ]:
# 4. Final ROI model: the selected method trained on all 40 train images
!python -m roi.run --final 2>&1 | grep -v -E "Warning|warn"

In [ ]:
# 5. Step 6.5 - ROI IoU on the 10 validation reference maps (+ lit mask, largest region, human ceiling)
from IPython.display import Image, display
!python -m roi.run --iou 2>&1 | grep -v -E "Warning|warn"
display(Image(f"{OUTPUT_ROOT}/roi/figures/val_roi.png", width=1100))

In [ ]:
# 6. Step 6.4 - ROI masks for all validation and calibration images
!python -m roi.run --apply --split val 2>&1 | grep -v -E "Warning|warn"
!python -m roi.run --apply --split calibration 2>&1 | grep -v -E "Warning|warn"

## Step 6.6 - Phases 3-5 inside the ROI (CPU work; existing segmentation masks)
Same code as Phases 3-5 with `--roi`; results go to separate `_roi` folders, the Phase 3-5 results stay unchanged.

In [ ]:
# 7. Phase 3 extraction inside the ROI (validation + calibration)
!python -m extraction.run_extract --split val --roi
!python -m extraction.run_extract --split calibration --roi
!python -m extraction.run_extract --split val --roi --compare

In [ ]:
# 8. Phase 4 inside the ROI: scale re-calibrated on the calibration images, checked on validation
!python -m calibration.scale --roi
!python -m calibration.scale --roi --check

In [ ]:
# 9. Phase 5 inside the ROI: biomarkers vs Konan (compare with the Phase 5 baseline)
!python -m biomarkers.validate --roi
!python -m biomarkers.validate --roi --compare

In [ ]:
# 10. Figures: Bland-Altman inside the ROI + overlays of the largest remaining ECD errors
import pandas as pd
!python -m biomarkers.validate --roi --figure
B = f"{OUTPUT_ROOT}/biomarkers/val/stardist_finetune_flat_roi"
display(Image(f"{B}/figures/bland_altman.png", width=1200))
for i in pd.read_csv(f"{B}/worst.csv")["image_id"]:
    display(Image(f"{OUTPUT_ROOT}/extraction/full_val_litnorm_roi/stardist_finetune_flat/figures/{i}.png", width=900))